# 1. Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Setup

In [2]:
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/raw/student_dropout_risk_2026.csv")
print(df.shape)
df.head()

(100000, 34)


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,weekly_work_hours,course_type,study_mode,semester,international_student,admission_grade,previous_qualification_grade,units_enrolled,units_passed,units_failed,gpa_semester_1,gpa_semester_2,attendance_rate,assignments_submitted_rate,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,24,STEM,Online,7,0,164.6,165.1,11,11,0,3.44,2.55,77.3,75.0,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,0,Arts & Humanities,Full-time,6,1,175.2,164.1,14,11,3,2.56,2.53,52.1,80.1,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,16,Arts & Humanities,Full-time,3,0,126.0,179.3,17,14,3,3.43,2.73,100.0,67.8,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,0,Business,Full-time,1,0,75.5,161.9,13,11,2,2.51,2.10,83.9,100.0,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,0,Health Sciences,Part-time,2,0,89.0,101.8,8,7,1,3.93,2.88,61.6,100.0,4,7.4,1,0,7,0,10,24,5.8,0


# 3. General structure of dataset

We can see some many points.
 - First, we have no missing values, so its means that the dataset has beed preprocessed or syntactically optimized
 - Second, we have 3 types: integer, strings and floats
 - Thrid, the target feature has a massive volume of data (10k)
 - Fourth, we have 34 features

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 34 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   person_id                     100000 non-null  int64  
 1   age                           100000 non-null  int64  
 2   gender                        100000 non-null  str    
 3   nationality                   100000 non-null  str    
 4   first_generation              100000 non-null  int64  
 5   disability_status             100000 non-null  int64  
 6   family_income_level           100000 non-null  str    
 7   scholarship_holder            100000 non-null  int64  
 8   tuition_up_to_date            100000 non-null  int64  
 9   part_time_job                 100000 non-null  int64  
 10  weekly_work_hours             100000 non-null  int64  
 11  course_type                   100000 non-null  str    
 12  study_mode                    100000 non-null  str    
 

As we saw previously, we have no missing values and no duplicates values

In [8]:
print("Missing values per columns:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nRaws duplicated:", df.duplicated().sum())
print("IDs duplicated:", df["person_id"].duplicated().sum())

Missing values per columns:
Series([], dtype: int64)

Raws duplicated: 0
IDs duplicated: 0


we can see that the 68% of students decided to dropout and 32% refuse that

In [9]:
print(df["dropout"].value_counts())
print()
print(df["dropout"].value_counts(normalize=True).round(3))

dropout
0    67996
1    32004
Name: count, dtype: int64

dropout
0    0.68
1    0.32
Name: proportion, dtype: float64


We seem various insights such as:
- The mean of ```age``` is 21.52 and the range goes from 15y/o to 30y/o
And mainly, a brief descriptions of features

In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
person_id,100000.0,50000.500000,28867.657797,1.0,25000.75,50000.50,75000.25,100000.0
age,100000.0,21.520840,3.399468,15.0,19.00,21.00,24.00,30.0
first_generation,100000.0,0.420130,0.493582,0.0,0.00,0.00,1.00,1.0
disability_status,100000.0,0.208770,0.406432,0.0,0.00,0.00,0.00,1.0
scholarship_holder,100000.0,0.589700,0.491891,0.0,0.00,1.00,1.00,1.0
tuition_up_to_date,100000.0,0.784860,0.410921,0.0,1.00,1.00,1.00,1.0
part_time_job,100000.0,0.481920,0.499676,0.0,0.00,0.00,1.00,1.0
weekly_work_hours,100000.0,8.744620,10.232842,0.0,0.00,0.00,18.00,40.0
semester,100000.0,3.617970,2.112763,1.0,2.00,3.00,5.00,8.0
international_student,100000.0,0.180940,0.384970,0.0,0.00,0.00,0.00,1.0


we can see the categoricals features

In [11]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts().head(10))
    print()

--- gender ---
gender
Female    50731
Male      47224
Other      2045
Name: count, dtype: int64

--- nationality ---
nationality
USA            29979
India          11994
China          10013
Brazil          6873
Mexico          6149
UK              6050
Nigeria         5921
Germany         4072
South Korea     4060
Philippines     3956
Name: count, dtype: int64

--- family_income_level ---
family_income_level
Middle    40213
Low       34809
High      24978
Name: count, dtype: int64

--- course_type ---
course_type
STEM                     30141
Business                 24986
Arts & Humanities        18024
Health Sciences          14941
Law & Social Sciences    11908
Name: count, dtype: int64

--- study_mode ---
study_mode
Full-time    65096
Part-time    21975
Online       12929
Name: count, dtype: int64



C:\Users\PC\AppData\Local\Temp\ipykernel_16788\4160255847.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


In [12]:
print("Estudiantes menores de 17 años:", (df["age"] < 17).sum())
print("Aprobadas + reprobadas > inscritas:",
      ((df["units_passed"] + df["units_failed"]) > df["units_enrolled"]).sum())

Estudiantes menores de 17 años: 7753
Aprobadas + reprobadas > inscritas: 0
